In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                            f1_score, classification_report, roc_auc_score,
                            confusion_matrix, cohen_kappa_score)
from imblearn.over_sampling import SMOTE
import warnings
import time

warnings.filterwarnings('ignore')

df = pd.read_csv('reast Cancer METABRIC.csv')

target_col = 'Tumor Stage'

original_count = len(df)
df = df[df[target_col] != 4].copy()
removed_count = original_count - len(df)
df = df.dropna(subset=[target_col])
leakage_features = [
    'Overall Survival (Months)',
    'Overall Survival Status',
    'Patient\'s Vital Status',
    'Relapse Free Status (Months)',
    'Relapse Free Status',
    'Nottingham prognostic index',
]

removed_leakage = [f for f in leakage_features if f in df.columns]
df = df.drop(columns=removed_leakage)

missing_rates = df.isnull().sum() / len(df)
high_missing = missing_rates[missing_rates > 0.8].index.tolist()
high_missing = [c for c in high_missing if c != target_col]
if high_missing:
    df = df.drop(columns=high_missing)
exclude_kw = ['stage', 'unnamed', 'patient id', 'patient\'s vital',
              'survival', 'relapse', 'status']
feature_cols = []
for col in df.columns:
    if col == target_col:
        continue
    if any(kw in col.lower() for kw in exclude_kw):
        continue
    if df[col].isnull().sum() / len(df) < 0.5:
        feature_cols.append(col)

X = df[feature_cols].copy()
y = df[target_col].copy()

# Handle missing values
for col in X.select_dtypes(include=[np.number]).columns:
    if X[col].isnull().any():
        X[col].fillna(X[col].median(), inplace=True)

for col in X.select_dtypes(include=['object']).columns:
    if X[col].isnull().any():
        mode = X[col].mode()[0] if len(X[col].mode()) > 0 else 'Unknown'
        X[col].fillna(mode, inplace=True)

# Label encoding
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

print(f"Feature matrix: {X.shape}")

# Feature selection
K_FEATURES = 15

selector = SelectKBest(score_func=mutual_info_classif, k=min(K_FEATURES, len(feature_cols)))
selector.fit(X, y)

feature_scores = pd.DataFrame({
    'feature': feature_cols,
    'score': selector.scores_
}).sort_values('score', ascending=False)

print(feature_scores.head(20).to_string(index=False))

selected_mask = selector.get_support()
selected_features = [feature_cols[i] for i in range(len(feature_cols)) if selected_mask[i]]

for i, feat in enumerate(selected_features, 1):
    score = feature_scores[feature_scores['feature'] == feat]['score'].values[0]
    print(f"  {i:2d}. {feat:<65} (Score: {score:.4f})")

X_selected = X[selected_features].copy()

selected_categorical = [col for col in selected_features if col in categorical_cols]
selected_numerical = [col for col in selected_features if col in numerical_cols]

# Data splitting
class_counts = y.value_counts()
print("\nNumber of samples per stage:")
for stage, count in class_counts.items():
    print(f"  Stage {stage}: {count} cases")

min_samples = class_counts.min()
stratify_param = None if min_samples < 2 else y

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y,
    test_size=0.3,
    random_state=42,
    stratify=stratify_param
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

print("\nTraining set distribution (before SMOTE):")
for s, c in y_train.value_counts().sort_index().items():
    print(f"  Stage {s}: {c} cases")


smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"\nBefore SMOTE: {X_train.shape}")
print(f"After SMOTE: {X_train_bal.shape}")
print(f"Added samples: {X_train_bal.shape[0] - X_train.shape[0]}")

print("\nTraining set distribution (after SMOTE):")
for s, c in pd.Series(y_train_bal).value_counts().sort_index().items():
    print(f"  Stage {s}: {c} cases")

# Target variable encoding (for neural networks)
target_encoder = LabelEncoder()
y_train_encoded = target_encoder.fit_transform(y_train_bal)
y_test_encoded = target_encoder.transform(y_test)
num_classes = len(target_encoder.classes_)

print(f"\nNumber of classes: {num_classes}")
print(f"Class mapping: {dict(zip(target_encoder.classes_, range(num_classes)))}")

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)


# FT-Transformer

class FTTransformer(nn.Module):

    def __init__(self,
                 num_features,
                 num_classes,
                 embedding_dim=33,
                 num_transformer_blocks=4,
                 num_heads=8,
                 dropout_rate=0.1,
                 ffn_dim=66):
        super(FTTransformer, self).__init__()

        self.num_features = num_features
        self.num_classes = num_classes
        self.embedding_dim = embedding_dim

        # 1. Feature Tokenizer: per-feature weight and bias parameters
        self.feature_weights = nn.Parameter(torch.randn(num_features, embedding_dim))
        self.feature_biases = nn.Parameter(torch.randn(num_features, embedding_dim))

        # 2. [CLS] Token
        self.cls_token = nn.Parameter(torch.randn(1, 1, embedding_dim))

        # 3. Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=ffn_dim,
            dropout=dropout_rate,
            activation='relu',
            batch_first=True,
            norm_first=False  # Post-norm
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_transformer_blocks
        )

        # 4. Parallel processing layers
        # DepthwiseConv1D
        self.depthwise_conv = nn.Conv1d(
            in_channels=embedding_dim,
            out_channels=embedding_dim,
            kernel_size=1,
            groups=1,
            padding=0
        )

        # Global Average Pooling
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)

        # LSTM
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=32,
            num_layers=1,
            batch_first=True,
            dropout=0.0
        )

        # 5. Final classifier
        concat_dim = (num_features * embedding_dim) + embedding_dim + 32

        self.classifier = nn.Sequential(
            nn.Linear(concat_dim, num_classes)
        )

    def forward(self, x):

        batch_size = x.size(0)

        # Feature Tokenization: T = b + x * W
        x_expanded = x.unsqueeze(-1)

        feature_embeddings = self.feature_biases.unsqueeze(0) + \
                           x_expanded * self.feature_weights.unsqueeze(0)

        # Add [CLS] token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)  # (B, 1, D)
        transformer_input = torch.cat([cls_tokens, feature_embeddings], dim=1)  # (B, F+1, D)

        # Transformer Encoding
        transformer_output = self.transformer(transformer_input)  # (B, F+1, D)

        # Remove [CLS] token, keep feature outputs only
        feature_output = transformer_output[:, 1:, :]  # (B, F, D)

        # 4.1 DepthwiseConv1D branch
        conv_input = feature_output.permute(0, 2, 1)
        conv_output = self.depthwise_conv(conv_input)  # (B, D, F)
        conv_output = conv_output.permute(0, 2, 1)  # (B, F, D)
        conv_flat = conv_output.reshape(batch_size, -1)  # (B, F*D)

        # 4.2 Global Average Pooling branch
        gap_input = feature_output.permute(0, 2, 1)  # (B, D, F)
        gap_output = self.global_avg_pool(gap_input)  # (B, D, 1)
        gap_flat = gap_output.squeeze(-1)  # (B, D)

        # 4.3 LSTM branch
        lstm_output, _ = self.lstm(feature_output)  # (B, F, 32)
        lstm_flat = lstm_output[:, -1, :]  # last time step (B, 32)

        # Concatenate all branches
        concatenated = torch.cat([conv_flat, gap_flat, lstm_flat], dim=1)  # (B, F*D+D+32)

        logits = self.classifier(concatenated)  # (B, num_classes)

        return logits


# Prepare PyTorch datasets

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = TabularDataset(X_train_scaled, y_train_encoded)
test_dataset = TabularDataset(X_test_scaled, y_test_encoded)

BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of test batches: {len(test_loader)}")

# Hyperparameters (based on paper Table 1)
HYPERPARAMS = {
    'embedding_dim': 16,
    'num_transformer_blocks': 4,
    'num_heads': 8,
    'dropout_rate': 0.10,
    'ffn_dim': 66,
    'learning_rate': 0.0001,
    'weight_decay': 0.0001,
    'num_epochs': 200,
    'batch_size': 64,
}

for param, value in HYPERPARAMS.items():
    print(f"  {param}: {value}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\ndevice: {device}")

model = FTTransformer(
    num_features=len(selected_features),
    num_classes=num_classes,
    embedding_dim=HYPERPARAMS['embedding_dim'],
    num_transformer_blocks=HYPERPARAMS['num_transformer_blocks'],
    num_heads=HYPERPARAMS['num_heads'],
    dropout_rate=HYPERPARAMS['dropout_rate'],
    ffn_dim=HYPERPARAMS['ffn_dim']
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    model.parameters(),
    lr=HYPERPARAMS['learning_rate'],
    weight_decay=HYPERPARAMS['weight_decay']
)

best_test_acc = 0.0
patience = 20
patience_counter = 0
train_start_time = time.time()

for epoch in range(HYPERPARAMS['num_epochs']):

    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_total += y_batch.size(0)
        train_correct += (predicted == y_batch).sum().item()

    train_acc = train_correct / train_total
    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            _, predicted = torch.max(outputs.data, 1)
            test_total += y_batch.size(0)
            test_correct += (predicted == y_batch).sum().item()

    test_acc = test_correct / test_total

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:3d}/{HYPERPARAMS['num_epochs']}] | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Train Acc: {train_acc:.4f} | "
              f"Test Acc: {test_acc:.4f}")

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        patience_counter = 0
        torch.save(model.state_dict(), 'best_fttransformer_breast_cancer.pth')
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

train_end_time = time.time()
training_time = train_end_time - train_start_time

print(f"Best test accuracy: {best_test_acc:.4f}")
print(f"Training time: {training_time:.2f} seconds ({training_time/60:.2f} minutes)")

model.load_state_dict(torch.load('best_fttransformer_breast_cancer.pth'))

# Cross-validation note: deep learning models require retraining for each fold
print(f"Current best test accuracy: {best_test_acc:.4f}")

# Feature importance analysis
model.eval()
train_features = []
train_labels = []

with torch.no_grad():
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)

        x_expanded = X_batch.unsqueeze(-1)
        feature_embeddings = model.feature_biases.unsqueeze(0) + \
                           x_expanded * model.feature_weights.unsqueeze(0)
        cls_tokens = model.cls_token.expand(X_batch.size(0), -1, -1)
        transformer_input = torch.cat([cls_tokens, feature_embeddings], dim=1)
        transformer_output = model.transformer(transformer_input)

        features_flat = transformer_output.reshape(X_batch.size(0), -1)

        train_features.append(features_flat.cpu().numpy())
        train_labels.append(y_batch.cpu().numpy())

train_features = np.vstack(train_features)
train_labels = np.hstack(train_labels)

rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf.fit(train_features, train_labels)

print(feature_scores[feature_scores['feature'].isin(selected_features)].to_string(index=False))

# Model evaluation

def evaluate_model(model, data_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)

            outputs = model(X_batch)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

    # AUC
    try:
        if num_classes == 2:
            auc = roc_auc_score(all_labels, all_probs[:, 1])
        else:
            auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='weighted')
    except:
        auc = 0.0

    # Specificity
    cm = confusion_matrix(all_labels, all_preds)
    if num_classes == 2:
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    else:
        specificity = 0.0
        for i in range(num_classes):
            tn = np.sum(cm) - (np.sum(cm[i, :]) + np.sum(cm[:, i]) - cm[i, i])
            fp = np.sum(cm[:, i]) - cm[i, i]
            specificity += tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specificity /= num_classes

    # Cohen's Kappa
    kappa = cohen_kappa_score(all_labels, all_preds)

    return accuracy, precision, recall, f1, specificity, kappa, auc, all_preds, all_labels


train_accuracy, train_precision, train_recall, train_f1, train_spec, train_kappa, train_auc, _, _ = evaluate_model(
    model, train_loader, device
)

test_accuracy, test_precision, test_recall, test_f1, test_spec, test_kappa, test_auc, y_test_pred, y_test_labels = evaluate_model(
    model, test_loader, device
)

print("=" * 80)
print(" " * 20 + "Model Performance Comparison")
print("=" * 80)
print(f"{'Metric':<15} {'Training Set':>15} {'Test Set':>15} {'Difference':>15}")
print("-" * 80)
print(f"{'Accuracy':<15} {train_accuracy*100:>14.2f}% {test_accuracy*100:>14.2f}% {abs(train_accuracy-test_accuracy)*100:>14.2f}%")
print(f"{'Precision':<15} {train_precision*100:>14.2f}% {test_precision*100:>14.2f}% {abs(train_precision-test_precision)*100:>14.2f}%")
print(f"{'Recall':<15} {train_recall*100:>14.2f}% {test_recall*100:>14.2f}% {abs(train_recall-test_recall)*100:>14.2f}%")
print(f"{'F1 Score':<15} {train_f1*100:>14.2f}% {test_f1*100:>14.2f}% {abs(train_f1-test_f1)*100:>14.2f}%")
print(f"{'Specificity':<15} {train_spec*100:>14.2f}% {test_spec*100:>14.2f}% {abs(train_spec-test_spec)*100:>14.2f}%")
print(f"{'Cohen Kappa':<15} {train_kappa*100:>14.2f}% {test_kappa*100:>14.2f}% {abs(train_kappa-test_kappa)*100:>14.2f}%")
print(f"{'AUC':<15} {train_auc*100:>14.2f}% {test_auc*100:>14.2f}% {abs(train_auc-test_auc)*100:>14.2f}%")
print("=" * 80)

y_test_original = target_encoder.inverse_transform(y_test_labels)
y_test_pred_original = target_encoder.inverse_transform(y_test_pred)
print(classification_report(y_test_original, y_test_pred_original, zero_division=0))